# Stormlight character interaction graph

Builds an undirected weighted graph from **scene-level co-occurrence**: two characters get an edge whenever they're named in the same scene (delimited by `* * *`). Edge weight = number of shared scenes.

Outputs:
- `character_graph.png` — static 3-panel figure (WoK / WoR / combined)
- `character_graph_timeline.html` — Plotly with a slider; drag to scrub through chapters and watch the network grow
- `character_graph_pyvis.html` — drag-and-explore full network (Pyvis)
- `character_graph.graphml` — for **Gephi** (the gold standard for arbitrary chapter-range filtering via Timeline)
- `csv_data/character_{nodes,edges,edges_per_scene}.csv` — graph data as CSVs

Character lexicon is derived from `stormlight_stopwords.txt` (the people-sections) + a few manual aliases (Kal→Kaladin, Hoid↔Wit, etc.).


In [ ]:
# ── Cell 2: Imports + config ─────────────────────────────────────────────────
import re, json
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

REPO          = Path("/Users/caputomachine/Desktop/StormlightCorpus")
PARAS_IN      = REPO / "csv_data" / "paragraphs.csv"
STOPWORDS_IN  = REPO / "stormlight_viz" / "stormlight_stopwords.txt"

NODES_OUT     = REPO / "csv_data" / "character_nodes.csv"
EDGES_OUT     = REPO / "csv_data" / "character_edges.csv"           # aggregated
PER_SCENE_OUT = REPO / "csv_data" / "character_edges_per_scene.csv" # raw log

PNG_OUT       = "character_graph.png"
TIMELINE_HTML = "character_graph_timeline.html"
PYVIS_HTML    = "character_graph_pyvis.html"
GRAPHML_OUT   = "character_graph.graphml"

MIN_EDGE_WEIGHT = 2   # drop edges with fewer than N shared scenes (visual noise filter)
LABEL_TOP_N     = 25  # only label the top-N highest-degree nodes on the static plot


In [ ]:
# ── Cell 3: Build character lexicon from stopwords + aliases ──────────────────
# Parse only the people-sections of the stopwords file. Lines under those
# section headers become canonical character names.

CHARACTER_SECTIONS = {
    "POV characters",
    "Major non-POV characters",
    "Bridge Four",          # partial-match against "Bridge Four / Kaladin's circle"
    "Alethi highprinces",   # partial-match against the highprinces header
    "Other Heralds",        # partial-match against the Heralds header
}

# Aliases (lowercase surface form → canonical title-cased name).
# Two characters share a canonical when they're the same person (e.g. Hoid/Wit).
ALIASES = {
    "kal":         "Kaladin",
    "stormblessed":"Kaladin",
    "sylphrena":   "Syl",
    "hoid":        "Wit",
    "veil":        "Shallan",
    "talenelat":   "Talenel",
    "taln":        "Talenel",
    "kelek":       "Kalak",
    "lunamor":     "Rock",
}

# House / family names that get mentioned ("Brightlord Kholin", "House Davar")
# but aren't a single person — exclude them from the character lexicon even
# though they're in the stopwords file.
NOT_PEOPLE = {"kholin", "davar"}

def parse_character_sections(path: Path) -> list[str]:
    names = []
    in_chars = False
    for line in open(path):
        stripped = line.strip()
        if stripped.startswith("#"):
            comment = stripped.lstrip("#").strip(" ─-")
            in_chars = any(s in comment for s in CHARACTER_SECTIONS)
            continue
        if not stripped:
            continue
        if in_chars:
            names.append(stripped.lower())
    return names

raw_names = [n for n in parse_character_sections(STOPWORDS_IN) if n not in NOT_PEOPLE]

# Surface form (lowercase) → canonical (title case).
SURFACE_TO_CANON: dict[str, str] = {}
for n in raw_names:
    canon = ALIASES.get(n, " ".join(w.capitalize() for w in n.split()))
    SURFACE_TO_CANON[n] = canon
# Ensure every alias key is also present
for surf, canon in ALIASES.items():
    SURFACE_TO_CANON.setdefault(surf, canon)

CANONICAL = sorted(set(SURFACE_TO_CANON.values()))
print(f"{len(SURFACE_TO_CANON)} surface forms → {len(CANONICAL)} canonical characters")
print(f"\nSample canonical names:")
for n in CANONICAL[:30]:
    print(f"  {n}")

# Build one big regex of all surface forms, longest first to prefer multi-word matches
surfaces_sorted = sorted(SURFACE_TO_CANON.keys(), key=len, reverse=True)
NAME_RE = re.compile(
    r"\b(" + "|".join(re.escape(s) for s in surfaces_sorted) + r")\b",
    re.IGNORECASE,
)
# Case-INSENSITIVE regex (patterns are lowercased); detect_chars() rejects matches
# whose first letter isn't capitalized, so "lift" the verb is filtered but "Lift"
# the character is kept.


In [ ]:
# ── Cell 4: Detect characters per scene ──────────────────────────────────────
paras = pd.read_csv(PARAS_IN)
print(f"paragraphs: {len(paras):,}")

# Group paragraphs into scenes; concatenate text per scene
scenes = (
    paras.groupby(["book", "chapter_order", "scene_idx", "heading_id", "pov"],
                  as_index=False)
         .agg(text=("text", lambda s: " ".join(s.astype(str))),
              n_paras=("text", "count"))
)
print(f"scenes:     {len(scenes):,}")

def detect_chars(text: str) -> set[str]:
    found: set[str] = set()
    for m in NAME_RE.finditer(text):
        surface = m.group(1)
        if not surface or not surface[0].isupper():
            continue   # case-sensitive: skip "lift" the verb, "rock" the noun, etc.
        canon = SURFACE_TO_CANON.get(surface.lower())
        if canon:
            found.add(canon)
    return found

scenes["characters"] = scenes["text"].map(detect_chars)
scenes["n_chars"]    = scenes["characters"].map(len)

# Sanity
print(f"\nscenes with ≥2 characters: {(scenes['n_chars'] >= 2).sum():,}")
print(f"top characters by scene mentions:")
mention_count = Counter()
for chars in scenes["characters"]:
    for c in chars:
        mention_count[c] += 1
for c, n in mention_count.most_common(15):
    print(f"  {n:>4}  {c}")


In [ ]:
# ── Cell 5: Build per-scene edge log + aggregated weighted edges ─────────────
rows = []
for _, s in scenes.iterrows():
    chars = sorted(s["characters"])
    if len(chars) < 2:
        continue
    for a, b in combinations(chars, 2):
        rows.append({
            "source":        a,
            "target":        b,
            "book":          s["book"],
            "chapter_order": s["chapter_order"],
            "scene_idx":     s["scene_idx"],
            "heading_id":    s["heading_id"],
            "pov":           s["pov"],
        })

edges_per_scene = pd.DataFrame(rows)
print(f"per-scene edges (raw): {len(edges_per_scene):,}")

# Aggregate to one row per (source, target, book) — weight = shared-scene count
edges_agg = (
    edges_per_scene.groupby(["source", "target", "book"], as_index=False)
                   .size()
                   .rename(columns={"size": "weight"})
)
print(f"aggregated edges (per book): {len(edges_agg):,}")
print(f"top edges by weight:")
print(edges_agg.sort_values("weight", ascending=False).head(12).to_string(index=False))

edges_per_scene.to_csv(PER_SCENE_OUT, index=False)
edges_agg.to_csv(EDGES_OUT, index=False)
print(f"\nwrote {PER_SCENE_OUT.name} and {EDGES_OUT.name}")


In [ ]:
# ── Cell 6: Build graphs + Louvain community detection ──────────────────────
import community as community_louvain  # python-louvain

def build_graph(edges: pd.DataFrame, min_w: int = MIN_EDGE_WEIGHT) -> nx.Graph:
    G = nx.Graph()
    e_filtered = edges[edges["weight"] >= min_w]
    for _, r in e_filtered.iterrows():
        if G.has_edge(r["source"], r["target"]):
            G[r["source"]][r["target"]]["weight"] += int(r["weight"])
        else:
            G.add_edge(r["source"], r["target"], weight=int(r["weight"]))
    return G

# Aggregate weights across books for the combined graph
edges_combined = (
    edges_per_scene.groupby(["source", "target"], as_index=False)
                   .size().rename(columns={"size": "weight"})
)

G_wok      = build_graph(edges_agg[edges_agg["book"] == "The Way of Kings"])
G_wor      = build_graph(edges_agg[edges_agg["book"] == "Words of Radiance"])
G_combined = build_graph(edges_combined)

print(f"WoK      : {G_wok.number_of_nodes()} nodes, {G_wok.number_of_edges()} edges")
print(f"WoR      : {G_wor.number_of_nodes()} nodes, {G_wor.number_of_edges()} edges")
print(f"Combined : {G_combined.number_of_nodes()} nodes, {G_combined.number_of_edges()} edges")

# Louvain communities on the combined graph
partition = community_louvain.best_partition(G_combined, random_state=42, weight="weight", resolution=1.5)
n_communities = len(set(partition.values()))
print(f"\nLouvain found {n_communities} communities on the combined graph")

# Save nodes CSV (one row per character) with mentions, communities, degrees
nodes_df = pd.DataFrame({
    "character": list(G_combined.nodes()),
    "mentions": [mention_count[n] for n in G_combined.nodes()],
    "community": [partition[n] for n in G_combined.nodes()],
    "degree_combined": [G_combined.degree(n) for n in G_combined.nodes()],
    "degree_wok":      [G_wok.degree(n) if G_wok.has_node(n) else 0 for n in G_combined.nodes()],
    "degree_wor":      [G_wor.degree(n) if G_wor.has_node(n) else 0 for n in G_combined.nodes()],
}).sort_values("mentions", ascending=False).reset_index(drop=True)
nodes_df.to_csv(NODES_OUT, index=False)
print(f"\nwrote {NODES_OUT.name}")
print(nodes_df.head(15).to_string(index=False))


In [ ]:
# ── Cell 7: Matplotlib static — single large combined figure ─────────────────
# Community-aware spring layout: seeds each Louvain community on its own arc of
# a circle, then runs spring with those initial positions so the global structure
# (which community is where) is stable while local edge attraction does the rest.

def community_aware_layout(G, partition, k=0.8, iterations=120, seed=42):
    communities = sorted(set(partition.values()))
    n_comm = len(communities)
    angle = 2 * np.pi / max(n_comm, 1)
    radius = 2.0
    centers = {c: (np.cos(i * angle) * radius, np.sin(i * angle) * radius)
               for i, c in enumerate(communities)}
    rng = np.random.RandomState(seed)
    initial_pos = {}
    for n in G.nodes():
        cx, cy = centers[partition.get(n, communities[0])]
        initial_pos[n] = (cx + rng.uniform(-0.4, 0.4), cy + rng.uniform(-0.4, 0.4))
    return nx.spring_layout(G, pos=initial_pos, k=k, iterations=iterations,
                            seed=seed, weight="weight")

fig, ax = plt.subplots(figsize=(18, 14))

pos = community_aware_layout(G_combined, partition, k=0.55, iterations=180, seed=42)

n_comm = len(set(partition.values()))
palette = plt.get_cmap("tab20", max(20, n_comm))
node_color = {n: palette(partition.get(n, 0) % 20) for n in G_combined.nodes()}

# Node sizes (log-scaled mention count so outliers don't drown out the rest)
mentions_lookup = {n: mention_count.get(n, 1) for n in G_combined.nodes()}
mlog = {n: np.log1p(m) for n, m in mentions_lookup.items()}
mmax = max(mlog.values()) if mlog else 1
node_sizes = [80 + 600 * (mlog[n] / mmax) for n in G_combined.nodes()]
node_colors_list = [node_color[n] for n in G_combined.nodes()]

# Edges — log-scale thickness, alpha proportional to relative weight
weights = np.array([G_combined[u][v]["weight"] for u, v in G_combined.edges()])
wlog = np.log1p(weights)
wmax = wlog.max() if len(wlog) else 1
widths = 0.3 + 2.8 * (wlog / wmax)
alphas = 0.10 + 0.55 * (weights / weights.max() if len(weights) else 1)
edge_colors = [(0.35, 0.35, 0.40, float(a)) for a in alphas]

nx.draw_networkx_edges(G_combined, pos, ax=ax, width=widths,
                       edge_color=edge_colors)
nx.draw_networkx_nodes(G_combined, pos, ax=ax, node_size=node_sizes,
                       node_color=node_colors_list, edgecolors="white",
                       linewidths=1.0)

# Two-tier labels: top-10 by weighted degree in bold, next 20 in regular
deg_weighted = dict(G_combined.degree(weight="weight"))
ranked = sorted(deg_weighted, key=deg_weighted.get, reverse=True)
tier1 = ranked[:10]
tier2 = ranked[10:30]

import matplotlib.patheffects as pe
for n in tier1:
    x, y = pos[n]
    t = ax.text(x, y, n, fontsize=11, fontweight="bold",
                ha="center", va="center", zorder=10)
    t.set_path_effects([pe.withStroke(linewidth=3, foreground="white")])
for n in tier2:
    x, y = pos[n]
    t = ax.text(x, y, n, fontsize=8.5, fontweight="regular",
                ha="center", va="center", zorder=9, color="#333")
    t.set_path_effects([pe.withStroke(linewidth=2, foreground="white")])

# Community legend
from matplotlib.lines import Line2D
comm_handles = [
    Line2D([0], [0], marker="o", linestyle="None",
           markerfacecolor=palette(c % 20), markeredgecolor="white",
           markersize=11, label=f"Community {c}  ({sum(1 for p in partition.values() if p == c)} chars)")
    for c in sorted(set(partition.values()))
]
ax.legend(handles=comm_handles, title="Louvain communities (resolution 1.5)",
          loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False, fontsize=9)

ax.set_title(
    f"Stormlight character interactions — scene co-occurrence  "
    f"(edges w ≥ {MIN_EDGE_WEIGHT}, log-scale thickness, community-aware layout)\n"
    f"n = {G_combined.number_of_nodes()} characters, m = {G_combined.number_of_edges()} edges",
    fontsize=14, fontweight="bold",
)
ax.axis("off")

plt.tight_layout()
plt.savefig(PNG_OUT, dpi=150, bbox_inches="tight")
plt.show()
print(f"saved {PNG_OUT}")


In [ ]:
# ── Cell 8: Plotly timeline-slider HTML (cumulative + windowed) ──────────────
# Slider scrubs through ~24 narrative checkpoints. Two mode buttons:
#   - "Cumulative":  show every interaction from chapter 1 up to current
#   - "Windowed":    show only interactions in the last WINDOW_SIZE chapters
# Both modes share the same node positions (computed from the full combined graph)
# so the eye can follow nodes across modes.

import plotly.graph_objects as go

# Narrative position = sequential index over all (book, chapter) tuples that have edges.
chapter_keys = (
    edges_per_scene[["book", "chapter_order"]].drop_duplicates()
    .sort_values(["book", "chapter_order"]).values.tolist()
)
narr_pos_map = {tuple(k): i + 1 for i, k in enumerate(chapter_keys)}
edges_per_scene["narr_pos"] = edges_per_scene.apply(
    lambda r: narr_pos_map[(r["book"], r["chapter_order"])], axis=1
)
all_positions = sorted(edges_per_scene["narr_pos"].unique())
last_pos      = all_positions[-1]
book_boundary = next(i for i, k in enumerate(chapter_keys, start=1)
                     if k[0] == "Words of Radiance")  # first WoR position

WINDOW_SIZE   = 15
N_CHECKPOINTS = 24
step = max(1, len(all_positions) // N_CHECKPOINTS)
checkpoints = all_positions[::step]
if checkpoints[-1] != last_pos:
    checkpoints.append(last_pos)

# Use the community-aware layout from the static cell so positions match
pos_combined = community_aware_layout(G_combined, partition, k=0.55,
                                       iterations=180, seed=42)

def slice_to_graph(slice_df):
    """Aggregate a per-scene slice into (edges_dict, nodes_set, degree_dict)."""
    if slice_df.empty:
        return {}, set(), {}
    pairs = (slice_df.groupby(["source", "target"], as_index=False)
                     .size().rename(columns={"size": "w"}))
    edges = {(r["source"], r["target"]): int(r["w"]) for _, r in pairs.iterrows()}
    nodes = set(slice_df["source"]) | set(slice_df["target"])
    deg = defaultdict(int)
    for (u, v), w in edges.items():
        deg[u] += w; deg[v] += w
    return edges, nodes, deg

def rgba_to_hex(c):
    return "#{:02x}{:02x}{:02x}".format(int(c[0]*255), int(c[1]*255), int(c[2]*255))

def build_traces(slice_df, label_suffix=""):
    edges, nodes, deg = slice_to_graph(slice_df)
    # Edges
    ex, ey = [], []
    for (u, v) in edges.keys():
        if u not in pos_combined or v not in pos_combined: continue
        x0, y0 = pos_combined[u]; x1, y1 = pos_combined[v]
        ex += [x0, x1, None]; ey += [y0, y1, None]
    edge_trace = go.Scatter(
        x=ex, y=ey, mode="lines",
        line=dict(color="rgba(120,120,120,0.35)", width=1.0),
        hoverinfo="skip", showlegend=False,
    )
    # Nodes
    names = [n for n in nodes if n in pos_combined]
    if not names:
        names = [list(pos_combined.keys())[0]]
    nx_x = [pos_combined[n][0] for n in names]
    nx_y = [pos_combined[n][1] for n in names]
    degs = [deg.get(n, 0) for n in names]
    sizes = [6 + 0.30 * d for d in degs]
    colors = [rgba_to_hex(palette(partition.get(n, 0) % 20)) for n in names]
    threshold = max(degs) * 0.25 if degs else 0
    labels = [n if d >= threshold else "" for n, d in zip(names, degs)]
    node_trace = go.Scatter(
        x=nx_x, y=nx_y, mode="markers+text",
        text=labels, textposition="top center", textfont=dict(size=10),
        marker=dict(size=sizes, color=colors, line=dict(color="white", width=1)),
        hovertext=[f"{n}<br>weighted degree: {d}" for n, d in zip(names, degs)],
        hoverinfo="text", showlegend=False,
    )
    return edge_trace, node_trace

# Build frames: each frame replaces all 4 traces with current state for both modes
def frame_for(np_pos):
    cum_slice = edges_per_scene[edges_per_scene["narr_pos"] <= np_pos]
    win_slice = edges_per_scene[
        (edges_per_scene["narr_pos"] > np_pos - WINDOW_SIZE) &
        (edges_per_scene["narr_pos"] <= np_pos)
    ]
    ce, cn = build_traces(cum_slice, " (cum)")
    we, wn = build_traces(win_slice, " (win)")
    return [ce, cn, we, wn]

def label_for(np_pos):
    book, ch = chapter_keys[np_pos - 1]
    short_book = "WoK" if book == "The Way of Kings" else "WoR"
    return f"{short_book} ch{int(ch)}"

print(f"building {len(checkpoints)} checkpoints × 2 modes ...")
frames = [go.Frame(data=frame_for(p), name=str(p)) for p in checkpoints]

# Initial figure shows the last cumulative frame (full network), windowed traces hidden
init = frames[-1].data
init[0].visible = True   # cum edges
init[1].visible = True   # cum nodes
init[2].visible = False  # win edges
init[3].visible = False  # win nodes

fig = go.Figure(data=init, frames=frames)

fig.update_layout(
    title=f"Stormlight character interactions — interactive timeline (window = {WINDOW_SIZE} chapters)",
    width=1100, height=780,
    plot_bgcolor="white",
    xaxis=dict(visible=False),
    yaxis=dict(visible=False, scaleanchor="x", scaleratio=1),
    sliders=[dict(
        active=len(checkpoints) - 1,
        currentvalue={"prefix": "Position: "},
        pad={"t": 30},
        steps=[dict(method="animate", label=label_for(p),
                    args=[[str(p)], dict(frame=dict(duration=0, redraw=True),
                                          mode="immediate", transition=dict(duration=0))])
               for p in checkpoints],
    )],
    updatemenus=[
        # Mode toggle: Cumulative vs Windowed
        dict(type="buttons", direction="right",
             x=0.50, y=1.10, xanchor="center", yanchor="bottom", showactive=True,
             buttons=[
                 dict(label="Cumulative", method="update",
                      args=[{"visible": [True, True, False, False]}]),
                 dict(label=f"Windowed (last {WINDOW_SIZE} ch.)", method="update",
                      args=[{"visible": [False, False, True, True]}]),
             ]),
        # Play / Pause for animation
        dict(type="buttons", direction="right",
             x=1.02, y=1.10, xanchor="right", yanchor="bottom", showactive=False,
             buttons=[
                 dict(label="▶ Play", method="animate",
                      args=[None, dict(frame=dict(duration=400, redraw=True),
                                       fromcurrent=True, mode="immediate")]),
                 dict(label="❚❚ Pause", method="animate",
                      args=[[None], dict(frame=dict(duration=0, redraw=False),
                                         mode="immediate")]),
             ]),
    ],
)
fig.write_html(TIMELINE_HTML, include_plotlyjs="cdn")
import os
print(f"saved {TIMELINE_HTML}  ({os.path.getsize(TIMELINE_HTML)/1024:.0f} KB)")
fig.show()


In [ ]:
# ── Cell 9: Pyvis drag-and-explore HTML (combined network) ──────────────────
from pyvis.network import Network

net = Network(height="800px", width="100%", bgcolor="white", font_color="#222",
              notebook=False, cdn_resources="in_line", directed=False)
net.barnes_hut(gravity=-12000, central_gravity=0.3, spring_length=120,
               spring_strength=0.01, damping=0.6, overlap=0.0)

for n in G_combined.nodes():
    wdeg = G_combined.degree(n, weight="weight")  # for tooltip
    deg  = G_combined.degree(n)                   # for size (unweighted)
    c = palette(partition.get(n, 0) % 20)
    color = "#{:02x}{:02x}{:02x}".format(int(c[0]*255), int(c[1]*255), int(c[2]*255))
    # Pyvis size: small range driven by unweighted degree, so top characters
    # are visible but don't swamp the canvas.
    net.add_node(n, label=n, title=f"{n}<br>weighted degree: {wdeg}<br>connections: {deg}",
                 size=10 + 0.45*deg, color=color)

max_w = max((d["weight"] for _, _, d in G_combined.edges(data=True)), default=1)
for u, v, d in G_combined.edges(data=True):
    w = d["weight"]
    net.add_edge(u, v, value=w, title=f"{u} ↔ {v}: {w} shared scenes",
                 width=0.5 + 3 * (w / max_w),
                 color={"color": "rgba(100,100,100,0.4)"})

net.write_html(PYVIS_HTML, notebook=False, open_browser=False)
print(f"saved {PYVIS_HTML}")


In [ ]:
# ── Cell 10: GraphML export + network-metrics report ─────────────────────────
# Write per-(source, target, chapter) edges so Gephi's Timeline filter can
# select arbitrary chapter ranges. Each edge has `book`, `chapter_order`, and
# `weight` attributes.

G_temporal = nx.MultiGraph()
for n in G_combined.nodes():
    G_temporal.add_node(
        n,
        mentions=int(mention_count.get(n, 0)),
        community=int(partition.get(n, 0)),
        degree=int(G_combined.degree(n)),
    )

scene_pair_counts = (
    edges_per_scene.groupby(["source", "target", "book", "chapter_order"], as_index=False)
                   .size().rename(columns={"size": "weight"})
)
for _, r in scene_pair_counts.iterrows():
    G_temporal.add_edge(
        r["source"], r["target"],
        book=str(r["book"]),
        chapter_order=int(r["chapter_order"]),
        global_order=book_offset[r["book"]] + int(r["chapter_order"]),
        weight=int(r["weight"]),
    )

nx.write_graphml(G_temporal, GRAPHML_OUT)
print(f"saved {GRAPHML_OUT}  ({G_temporal.number_of_nodes()} nodes, {G_temporal.number_of_edges()} time-stamped edges)")
print(f"\n→ Open in Gephi. Use Window → Timeline; set the time interval column to 'chapter_order' or 'global_order' to filter.")

# Centrality rankings
print("\n=== Top 10 by weighted degree (combined) ===")
deg = dict(G_combined.degree(weight="weight"))
for n, d in sorted(deg.items(), key=lambda kv: -kv[1])[:10]:
    print(f"  {d:>4}  {n}")

print("\n=== Top 10 by betweenness centrality (combined) ===")
bet = nx.betweenness_centrality(G_combined, weight="weight")
for n, b in sorted(bet.items(), key=lambda kv: -kv[1])[:10]:
    print(f"  {b:.3f}  {n}")

print("\n=== Top 10 by eigenvector centrality (combined) ===")
try:
    eig = nx.eigenvector_centrality_numpy(G_combined, weight="weight")
    for n, e in sorted(eig.items(), key=lambda kv: -kv[1])[:10]:
        print(f"  {e:.3f}  {n}")
except Exception as ex:
    print(f"  (eigenvector skipped: {ex})")
